# Workflow with Checks

1. LLM parses user intent, titles, hard filters.
2. Work IDs extracted from normalized correct titles.

In [7]:
import json, os, re
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from rapidfuzz import fuzz, process
from google import genai
from google.genai import types

PROJECT_ROOT = Path.cwd().parent
load_dotenv(PROJECT_ROOT / ".env")

INTERIM = PROJECT_ROOT / "data" / "interim"

MODEL = "gemini-3.5-flash"
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

BOOKS = pd.read_parquet(
    INTERIM / "english_works_all.parquet",
    columns=["work_id", "title", "year_first", "pages_median", "ratings_total"],
)
print(f"{len(BOOKS):,} works")

1,002,607 works


In [8]:
def normalize_title(title):
    '''
    params:
        title: raw title, from the catalog or from the llm
    returns lowercase, punctuation-free, with any "(Series, #N)" suffix removed
    '''
    title = re.sub(r"\s*\([^)]*#\d+\)\s*$", "", str(title))
    title = re.sub(r"[^\w\s]", " ", title.lower())
    return re.sub(r"\s+", " ", title).strip()


BOOKS["title_norm"] = BOOKS.title.map(normalize_title)
TITLES = BOOKS.title_norm.tolist()          # built once, reused every lookup
BOOKS[["title", "title_norm"]].head()

,title,title_norm
0,"The Hunger Games (The Hunger Games, #1)",the hunger games
1,Harry Potter and the Sorcerer's Stone (Harry P...,harry potter and the sorcerer s stone
2,"Twilight (Twilight, #1)",twilight
3,To Kill a Mockingbird,to kill a mockingbird
4,The Great Gatsby,the great gatsby


In [17]:
BOOKS = BOOKS.sort_values("ratings_total", ascending=False).reset_index(drop=True)
TITLES = BOOKS.title_norm.tolist()

In [9]:
PROMPT = """
You are a natural language extraction system. Understand the user's query and return JSON in exactly this format:

{
   "semantic": [],
   "title_mentions": [],
   "proposed_titles": [],
   "filters": {
                  "max_pages": null,
                  "min_pages": null,
                  "min_year": null,
                  "author_name": null,
                  "exclude_books": null,
                  "exclude_authors": null
               }
}

Rules:

1. semantic: only mood, style, theme or reading-preference signals the user stated. Examples: "feel-good", "easy to read", "dark". Never infer these from a book the user mentions. Exclude titles, author names and numbers.
2. title_mentions: book titles as the user typed them, misspellings preserved. "litle prince" -> "litle prince".
3. proposed_titles: your best guess at the canonical title for each mention, same order. A downstream catalog resolver validates these.
4. filters: only what the user stated. Unspecified filters are null. "short" -> max_pages 200. "long" -> min_pages 250.
5. Never ask about ambiguous titles. Title resolution happens downstream.
Return only valid JSON.
"""


def extract_query(user_query):
    '''
    params:
        user_query: raw user message
    returns the parsed json dict from the model
    '''
    response = client.models.generate_content(
        model=MODEL,
        contents=user_query,
        config=types.GenerateContentConfig(
            system_instruction=PROMPT,
            response_mime_type="application/json",
            temperature=0,
        ),
    )
    return json.loads(response.text)

In [10]:
def resolve_title(proposed, cutoff=70):
    '''
    params:
        proposed: the llm's canonical guess, e.g. "The Little Prince"
        cutoff: minimum score to accept a match
    returns {title, work_id, score} or None if nothing scored above cutoff
    '''
    query = normalize_title(proposed)
    match = process.extractOne(query, TITLES, scorer=fuzz.WRatio,
                               score_cutoff=cutoff)
    if match is None:
        return None

    row = BOOKS.iloc[match[2]]
    return {"title": row.title, "work_id": row.work_id, "score": round(match[1], 1)}


def finalize(parsed):
    '''
    params:
        parsed: output of extract_query
    returns the json handed to retrieval; unmatched titles are dropped
    '''
    matches = [m for t in parsed["proposed_titles"] if (m := resolve_title(t))]
    return {
        "semantic": parsed["semantic"],
        "titles":   [m["title"] for m in matches],
        "work_ids": [m["work_id"] for m in matches],
        "filters":  parsed["filters"],
    }

In [11]:
parsed = extract_query("something like the litle prince, easy to read and short")
finalize(parsed)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


{'semantic': ['easy to read'],
 'titles': ['The Little Prince'],
 'work_ids': ['2180358'],
 'filters': {'max_pages': 200,
  'min_pages': None,
  'min_year': None,
  'author_name': None,
  'exclude_books': None,
  'exclude_authors': None}}

In [16]:
BOOKS[BOOKS.title_norm == "the little prince"][
    ["work_id", "title", "year_first", "pages_median", "ratings_total"]
].sort_values("ratings_total", ascending=False).head()

,work_id,title,year_first,pages_median,ratings_total
70,2180358,The Little Prince,1943.0,99.0,878752
21308,50329544,The Little Prince,2008.0,110.0,5251
